# Notebook 02: API-function fingerprints and the two-stage hierarchy

**Class 04_01 (part 2).** The Windows Malware Detection Dataset counts occurrences of 21,918 API functions per
file (a binary fingerprint). Labels: 0 = benign, 1 = RedLineStealer, 2 = Downloader, 3 = RAT,
4 = BankingTrojan, 5 = SnakeKeyLogger, 6 = Spyware. Two ideas:

1. **Flat multiclass** — one model, 7 classes. Rare and hard families get buried.
2. **Two-stage hierarchy** — first *is this malicious at all?* (stage 1), then *which family?* only on the
   predicted-malicious files (stage 2). Each stage is an easier problem.

The matrix is 29,505 x 21,918 with 0.3% density: loading it eagerly kills the machine, so we scan and cast
chunks lazily with polars (the pattern any real 10k+ column dataset needs).

In [ ]:
import sys

import numpy as np
from scipy import sparse as sp

sys.path[:0] = [".", "..", "../../src", "../../../src"]
import aasplot as ap
import malwlib as mw

ap.setup("04")
# First run: ~9 min scan of 21,918 columns. The demo caches the sampled matrix so reruns are instant
# and the notebook never holds more than ~6 MB of data in memory.
CACHE = mw.ROOT / ".cache_api_sub"  # npz cache in the repo root, git-ignored
if (CACHE.with_suffix(".npz")).exists():
    M, y = sp.load_npz(CACHE.with_suffix(".npz")), np.load(CACHE.with_suffix(".npy"))
else:
    M, y = mw.load_api_sparse()
    sp.save_npz(CACHE.with_suffix(".npz"), M)
    np.save(CACHE.with_suffix(".npy"), y)
print(
    f"API matrix (sampled): {M.shape[0]:,} files x {M.shape[1]:,} API columns, density {M.nnz / (M.shape[0] * M.shape[1]):.4f}"
)

In [ ]:
import polars as pl

FAMS = mw.FAMILIES
counts = pl.DataFrame({"Type": y}).group_by("Type").len().sort("Type")
print(counts)
mean_calls = {t: float(M[np.where(y == t)[0]].sum(1).mean()) for t in range(7)}
print({FAMS[k]: round(v, 1) for k, v in mean_calls.items()})

## Flat multiclass baseline

A random forest straight on 7 classes. Watch the per-family F1: the common families are fine, the stealthy
small ones (Spyware makes ~10 API calls per file on average) collapse.

In [ ]:
from scipy import sparse as sp
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

itr, ite = train_test_split(np.arange(len(y)), test_size=0.25, stratify=y, random_state=0)
flat = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(M[itr], y[itr])
pf = flat.predict(M[ite])
print(classification_report(y[ite], pf, digits=3))
flat_macro = f1_score(y[ite], pf, average="macro")
print(f"flat macro-F1 {flat_macro:.4f}")

## Stage 1: benign vs malware (binary)

1,877 benign vs 27,628 malware — the *real* imbalance problem of this dataset, solved with the same forest.

In [ ]:
yb = (y > 0).astype(int)
rf1 = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(M[itr], yb[itr])
pb = rf1.predict_proba(M[ite])[:, 1]
mw.eval_bin(yb[ite], pb, "stage 1: ")

## Stage 2: family on predicted-malicious files

Stage 2 trains on the true malicious training files only; at test time it sees stage 1's positive predictions,
so its errors also carry stage 1's false positives.

In [ ]:
fam = y - 1
tr_mal = itr[fam[itr] >= 0]
rf2 = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(M[tr_mal], fam[tr_mal])
mal_idx = ite[pb > 0.5]
gold, pred_f = fam[mal_idx], rf2.predict(M[mal_idx])
print(f"stage 2 families (n={len(mal_idx):,}): macro-F1 {f1_score(gold, pred_f, average='macro'):.4f}")
print(classification_report(gold, pred_f, digits=3, zero_division=0))

In [ ]:
pred = np.where(pb > 0.5, rf2.predict(M[ite]) + 1, 0)  # stage 2 output +1 back to the 0..6 label space
hier_acc = (pred == y[ite]).mean()
hier_macro = f1_score(y[ite], pred, average="macro")
print(f"hierarchical end-to-end acc {hier_acc:.4f} macro-F1 {hier_macro:.4f}  vs flat {flat_macro:.4f}")

In [ ]:
f, ax = ap.fig("full", 1.9)
idx = np.arange(7)
w = 0.38
rep_f = f1_score(y[ite], pred, average=None)
rep_l = f1_score(y[ite], pf, average=None)
ax.bar(idx - w / 2, rep_l, w, color=ap.GREY, label="flat")
ax.bar(idx + w / 2, rep_f, w, color=ap.BLUE, label="hierarchical")
ax.set_xticks(idx, [f"{i} {FAMS[i][:7]}" for i in idx], rotation=30, ha="right")
ax.set_ylabel("F1")
ax.set_ylim(0, 1.25)
ax.legend(ncol=2, loc="upper center")
ap.save(f, "d04_flat_vs_hier")

## Takeaways

- The API-fingerprint matrix is the genuinely hard static problem: macro-F1 .647 flat on a 20,000-file
  stratified sample (the full 29,505-row scan peaks near 1.5 GB RAM; the demo reads 100-row UInt8 chunks
  and caches the ~6 MB sample). Snake (F1 .177) and Spyware (.460) are confused with each other because
  they make ~13 and ~10 calls where RATs make ~79.
- The two-stage hierarchy loses a little (macro-F1 .629 against .647) but names the family: stage 1 AUC .702,
  and the Snake/Spyware pair is confused again with the roles swapped (.505 and .072).
- Both stages inherit imbalance: stage 1 has 6.8% benign, stage 2 has rare families — the same
  threshold/weight choices as notebook 01 apply.